# Morgan-fingerprint chemotypes of the whole QM9

Companion of `qm9_chemotype_clustering.py`: ECFP4 (binary, 2048 bits) → L2 → PCA(50) → k-means(50, 10 initialisations),
seed 42, fitted once on all 128,908 QM9 molecules → `qm9_chemotypes.pkl.gz`. The shipped labels define the splits:
near-tied k-means optima depend on the BLAS thread count (re-runs gave ARI 0.66–0.9996), so `RECOMPUTE = True` writes
`qm9_chemotypes_rerun.pkl.gz` and reports its agreement instead of overwriting.

In [ ]:
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import qm9_chemotype_clustering as qcc        # this directory
from qtnet.data_utils import _balanced_group_partition

RECOMPUTE = False                               # True: re-run ECFP -> PCA -> k-means (thread-count dependent)
CHEMOTYPES_FILE = os.path.join(qcc.MOLECULAR_DIR, 'qm9_chemotypes.pkl.gz')           # shipped
RERUN_FILE = os.path.join(qcc.MOLECULAR_DIR, 'qm9_chemotypes_rerun.pkl.gz')          # RECOMPUTE output
RUNS_DIR = os.path.join(qcc.REPO_ROOT, 'experiments', 'molecular', 'env_holdout')

frame = qcc.load_qm9_classification_frame(os.path.join(qcc.MOLECULAR_DIR, 'aimel_clustered_molecular.pkl'),
                                          os.path.join(qcc.MOLECULAR_DIR, 'qm9_filtered.pkl'),
                                          os.path.join(qcc.MOLECULAR_DIR, 'qm9_classification_order.json.gz'))
print(f"{len(frame):,} molecules ({int(frame['in_aimel'].sum()):,} AIMEl)")

## 1. Clustering

In [ ]:
if RECOMPUTE:
    from sklearn.metrics import adjusted_rand_score
    from sklearn.preprocessing import normalize

    X, valid = qcc.morgan_matrix(frame['smiles'].tolist(), radius=2, n_bits=2048)
    lv, var = qcc.pca_kmeans(normalize(X), n_components=50, k=50, seed=42)
    lab = np.full(len(frame), -1, dtype=int)
    lab[np.where(valid)[0]] = lv
    pd.DataFrame({'index': frame['index'].to_numpy(), 'fp_cluster': [f'fp_{int(k)}' for k in lab]}).to_pickle(RERUN_FILE)
    shipped = pd.read_pickle(CHEMOTYPES_FILE)
    print(f"PCA keeps {var:.3f} of the variance | adjusted Rand index vs shipped: "
          f"{adjusted_rand_score(shipped['fp_cluster'], [f'fp_{int(k)}' for k in lab]):.4f}")
else:
    print('RECOMPUTE = False: using the shipped chemotypes')

## 2. Chemotype sizes

In [ ]:
chem = pd.read_pickle(CHEMOTYPES_FILE)
assert chem['index'].is_unique and set(chem['index']) == set(frame['index'])
m = frame[['index', 'in_aimel']].merge(chem, on='index', validate='one_to_one')
sizes = (m.assign(set=np.where(m['in_aimel'], 'AIMEl', 'QM9 remainder'))
          .groupby(['fp_cluster', 'set']).size().unstack('set', fill_value=0))
total = sizes.sum(axis=1)
sizes = sizes.loc[total.sort_values(ascending=False).index]
print(f"{len(sizes)} chemotypes, {total.min():,}–{total.max():,} molecules "
      f"(median {int(total.median()):,}); unparsed SMILES: {int((chem['fp_cluster'] == 'fp_-1').sum())}")
display(sizes[sizes['AIMEl'] <= 2])

In [ ]:
# ── style of the figures (QT-Net's published figures: analysis/inference/analyse_inference.py) ──
plt.rcParams.update({'font.family': 'sans-serif',
                     'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica', 'Liberation Sans'],
                     'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False})
FONTS = {'title': 18, 'label': 16, 'tick': 12, 'legend': 12}
TITLE_WEIGHT, LABEL_WEIGHT = 'normal', 'normal'
SET_COLORS = {'AIMEl': '#1a4f9c', 'QM9 remainder': '#e8604c'}    # as in the t-SNE maps of analysis/molecular

XLABEL, YLABEL = 'Chemotype (sorted by size)', 'Molecules'

fig, ax = plt.subplots(figsize=(16, 5.5), layout='constrained')
x = np.arange(len(sizes))
bottom = np.zeros(len(sizes))
for st, col in SET_COLORS.items():
    ax.bar(x, sizes[st], bottom=bottom, color=col, width=0.85, label=st)
    bottom += sizes[st].to_numpy()
ax.set_xticks(x)
ax.set_xticklabels([c.split('_')[1] for c in sizes.index], fontsize=FONTS['tick'] - 3)
ax.tick_params(axis='y', labelsize=FONTS['tick'])
ax.set_xlim(-0.7, len(sizes) - 0.3)
ax.set_xlabel(XLABEL, fontsize=FONTS['label'], fontweight=LABEL_WEIGHT)
ax.set_ylabel(YLABEL, fontsize=FONTS['label'], fontweight=LABEL_WEIGHT)
ax.legend(frameon=False, fontsize=FONTS['legend'])
plt.show()

## 2b. Acceptance gate: do chemotypes separate the targets?

η² of each target on AIMEl against label-shuffled chemotypes (20 shuffles).

In [ ]:
lab_int = chem.set_index('index').loc[frame['index'], 'fp_cluster'].str.split('_').str[1].astype(int).to_numpy()
gate = qcc.eta2_gate(frame, lab_int, seed=42, shuffles=20)
gate.round(4)

## 3. The five chemotype groups of the cross-validation

Recomputed for the five repeats and checked against the held-out chemotypes in the runs' `config.json`; the table
shows repeat 0.

In [ ]:
aimel = pd.read_pickle(os.path.join(qcc.MOLECULAR_DIR, 'aimel_clustered_molecular.pkl'))[['index']]
aimel = aimel.assign(index=aimel['index'].astype(np.int64)).merge(chem, on='index', how='left', validate='one_to_one')
assert aimel['fp_cluster'].notna().all()
gid = aimel['fp_cluster'].to_numpy(object)
groups = {r: _balanced_group_partition(gid, np.ones(len(gid), bool), 5, np.random.default_rng(42 + r)) for r in range(5)}

rows = []
for c in range(5):
    sel = groups[0] == c
    rows.append({'group': c, 'molecules': int(sel.sum()), 'chemotypes': ', '.join(sorted(set(gid[sel]), key=lambda s: int(s[3:])))})
display(pd.DataFrame(rows).set_index('group'))
for r in range(1, 5):
    moved = groups[r] != groups[0]
    print(f'repeat {r}: {int(moved.sum()):,} molecules in another group than in repeat 0 '
          f'(chemotypes {", ".join(sorted(set(gid[moved]), key=lambda s: int(s[3:]))) or "none"})')

checked = 0
for r in range(5):
    for c in range(5):
        cfg_path = os.path.join(RUNS_DIR, 'blind', f'fold_{5 * r + c}', 'frac_1.0', 'config.json')
        if os.path.exists(cfg_path):
            cfg = json.load(open(cfg_path))
            assert sorted(cfg['heldout_chemotypes']) == sorted(set(gid[groups[r] == c])), (r, c)
            checked += 1
print(f'held-out chemotypes equal the runs\' config.json in {checked}/25 folds')